### Phase 2 - Frequency-Domain Manipulation

Core frequency pipeline:
1. FFT transform
2. High-frequency suppression
3. Checkerboard-notch removal
4. Low-pass filtering
5. Slight frequency masking

#### Output
- Confidence progression across stages (mean +/- SEM)
- Paired statistical summary with essential fields:
  - `p_value`
  - `cohen_dz`
  - `relative_drop_percent`
  - `n_pairs`
- Radial energy analysis (low/mid/high frequency ratios)
- Band-wise ablation (remove low/mid/high frequency bands)
- Real vs Synthetic comparison (original vs low-pass)



In [65]:

# Imports and global setup
from __future__ import annotations

import random
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from scipy import stats

import torch
import torch.nn as nn
from torchvision import models, transforms


In [66]:

# Config: paths, constants, and experiment knobs
PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'Phase2' else Path.cwd().resolve()
PHASE2_DIR = PROJECT_ROOT / 'Phase2'
DATASET_ROOT = PROJECT_ROOT / 'dataset'
METADATA_PATH = DATASET_ROOT / 'metadata.csv'

OUT_DIR = PHASE2_DIR / 'results_freq'
SPECTRUM_DIR = OUT_DIR / 'spectrum'
MODIFIED_DIR = OUT_DIR / 'modified_images'

CIFAR_MEAN = [0.4914, 0.4822, 0.4465]
CIFAR_STD = [0.2023, 0.1994, 0.2010]
STAGES = ['original', 'suppress_high', 'remove_checkerboard', 'low_pass', 'slight_mask']
ABLATION_EXPERIMENTS = ['remove_high', 'remove_mid', 'remove_low']

IMG_SIZE = 32
NUM_SAMPLES_PER_CLASS = 24
TARGET_SPLIT = 'test'
TARGET_CLASSES = ['FAKE', 'REAL']
SEED = 42

LOW_BAND_MAX = 0.33
MID_BAND_MAX = 0.66

@dataclass
class FreqConfig:
    hf_cutoff: float = 0.55
    hf_strength: float = 0.75
    checker_depth: float = 0.65
    checker_sigma: float = 2.5
    lowpass_radius: float = 0.38
    slight_strength: float = 0.12

CFG = FreqConfig()

SPECTRUM_DIR.mkdir(parents=True, exist_ok=True)
MODIFIED_DIR.mkdir(parents=True, exist_ok=True)

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'DATASET_ROOT: {DATASET_ROOT}')
print(f'OUT_DIR: {OUT_DIR}')


PROJECT_ROOT: C:\Users\Vedha\feb hack\Feb-2026-Hackathon
DATASET_ROOT: C:\Users\Vedha\feb hack\Feb-2026-Hackathon\dataset
OUT_DIR: C:\Users\Vedha\feb hack\Feb-2026-Hackathon\Phase2\results_freq


In [67]:

# Model loading helpers (matches Phase1 architecture)
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def build_resnet18(num_classes: int = 2) -> nn.Module:
    model = models.resnet18(weights=None)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(256, num_classes),
    )
    return model


def find_checkpoint(project_root: Path) -> Path:
    candidates = [
        project_root / 'Phase1' / 'models' / 'resnet18_subset_best.pt',
        project_root / 'Phase1' / 'cifake_resnet18_final.pt',
        project_root / 'dataset' / 'models' / 'resnet18_subset_best.pt',
        project_root / 'dataset' / 'models' / 'cifake_resnet18_final.pt',
    ]
    for p in candidates:
        if p.exists():
            return p
    expected = '\n'.join(str(p) for p in candidates)
    raise FileNotFoundError(f'No checkpoint found. Expected one of:\n{expected}')


def model_transform(img_size: int) -> transforms.Compose:
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD),
    ])


def load_model(project_root: Path, device: torch.device) -> nn.Module:
    ckpt_path = find_checkpoint(project_root)
    model = build_resnet18(num_classes=2)
    checkpoint = torch.load(ckpt_path, map_location='cpu')
    state_dict = checkpoint['model_state_dict'] if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint else checkpoint
    model.load_state_dict(state_dict, strict=True)
    model.to(device).eval()
    print(f'Loaded checkpoint: {ckpt_path}')
    return model


In [68]:

# Frequency-domain operations + quality metrics
def load_image(path: Path, img_size: int) -> np.ndarray:
    img = Image.open(path).convert('RGB').resize((img_size, img_size), Image.Resampling.BILINEAR)
    arr = np.asarray(img, dtype=np.float32) / 255.0
    return np.clip(arr, 0.0, 1.0)


def rgb_fft(image: np.ndarray) -> np.ndarray:
    return np.fft.fftshift(np.fft.fft2(image, axes=(0, 1)), axes=(0, 1))


def rgb_ifft(freq: np.ndarray) -> np.ndarray:
    recon = np.fft.ifft2(np.fft.ifftshift(freq, axes=(0, 1)), axes=(0, 1)).real
    return np.clip(recon, 0.0, 1.0)


def spectrum_image(freq: np.ndarray) -> np.ndarray:
    mag = np.log1p(np.abs(freq)).mean(axis=2)
    mag -= mag.min()
    denom = mag.max() if mag.max() > 0 else 1.0
    return mag / denom


def radial_map(h: int, w: int) -> tuple[np.ndarray, float, int, int]:
    cy, cx = h // 2, w // 2
    y, x = np.ogrid[:h, :w]
    r = np.sqrt((y - cy) ** 2 + (x - cx) ** 2)
    rmax = np.sqrt(cy ** 2 + cx ** 2)
    return r, rmax, cy, cx


def high_frequency_suppression_mask(h: int, w: int, cutoff_ratio: float, strength: float) -> np.ndarray:
    r, rmax, _, _ = radial_map(h, w)
    r0 = cutoff_ratio * rmax
    decay = np.clip((r - r0) / max(rmax - r0, 1e-6), 0.0, 1.0)
    return np.clip(1.0 - strength * decay, 0.0, 1.0)


def checkerboard_notch_mask(h: int, w: int, depth: float, sigma: float) -> np.ndarray:
    _, _, cy, cx = radial_map(h, w)
    yy, xx = np.mgrid[0:h, 0:w]
    centers = [
        (cy + int(0.25 * h), cx), (cy - int(0.25 * h), cx),
        (cy, cx + int(0.25 * w)), (cy, cx - int(0.25 * w)),
        (cy + int(0.25 * h), cx + int(0.25 * w)), (cy + int(0.25 * h), cx - int(0.25 * w)),
        (cy - int(0.25 * h), cx + int(0.25 * w)), (cy - int(0.25 * h), cx - int(0.25 * w)),
    ]
    mask = np.ones((h, w), dtype=np.float32)
    for y0, x0 in centers:
        gauss = np.exp(-((yy - y0) ** 2 + (xx - x0) ** 2) / (2.0 * sigma * sigma))
        mask *= 1.0 - depth * gauss
    return np.clip(mask, 0.0, 1.0)


def low_pass_mask(h: int, w: int, radius_ratio: float) -> np.ndarray:
    r, rmax, _, _ = radial_map(h, w)
    return (r <= radius_ratio * rmax).astype(np.float32)


def slight_frequency_mask(h: int, w: int, strength: float, rng: np.random.Generator) -> np.ndarray:
    r, rmax, _, _ = radial_map(h, w)
    mask = np.ones((h, w), dtype=np.float32)
    high_band = r > 0.65 * rmax
    mask[high_band] -= strength * rng.random(np.count_nonzero(high_band), dtype=np.float32)
    return np.clip(mask, 0.0, 1.0)


def apply_frequency_pipeline(image: np.ndarray, cfg: FreqConfig, rng: np.random.Generator) -> dict[str, np.ndarray]:
    h, w, _ = image.shape
    freq0 = rgb_fft(image)

    f1 = freq0 * high_frequency_suppression_mask(h, w, cfg.hf_cutoff, cfg.hf_strength)[:, :, None]
    f2 = f1 * checkerboard_notch_mask(h, w, cfg.checker_depth, cfg.checker_sigma)[:, :, None]
    f3 = f2 * low_pass_mask(h, w, cfg.lowpass_radius)[:, :, None]
    f4 = f3 * slight_frequency_mask(h, w, cfg.slight_strength, rng)[:, :, None]

    return {
        'original': image,
        'suppress_high': rgb_ifft(f1),
        'remove_checkerboard': rgb_ifft(f2),
        'low_pass': rgb_ifft(f3),
        'slight_mask': rgb_ifft(f4),
        'spectrum_original': spectrum_image(freq0),
        'spectrum_modified': spectrum_image(f4),
    }


def radial_band_masks(h: int, w: int, low_max: float = LOW_BAND_MAX, mid_max: float = MID_BAND_MAX) -> dict[str, np.ndarray]:
    r, rmax, _, _ = radial_map(h, w)
    rr = r / (rmax + 1e-8)
    low = rr <= low_max
    mid = (rr > low_max) & (rr <= mid_max)
    high = rr > mid_max
    return {'low': low, 'mid': mid, 'high': high}


def radial_energy_bands(image: np.ndarray) -> dict[str, float]:
    freq = rgb_fft(image)
    power = (np.abs(freq) ** 2).mean(axis=2)
    masks = radial_band_masks(*power.shape)

    low_e = float(power[masks['low']].sum())
    mid_e = float(power[masks['mid']].sum())
    high_e = float(power[masks['high']].sum())
    total = low_e + mid_e + high_e + 1e-12

    return {
        'energy_low': low_e,
        'energy_mid': mid_e,
        'energy_high': high_e,
        'energy_low_ratio': low_e / total,
        'energy_mid_ratio': mid_e / total,
        'energy_high_ratio': high_e / total,
    }


def apply_band_ablation(image: np.ndarray, remove_band: str) -> np.ndarray:
    freq = rgb_fft(image)
    h, w, _ = freq.shape
    masks = radial_band_masks(h, w)
    keep_mask = (~masks[remove_band]).astype(np.float32)
    return rgb_ifft(freq * keep_mask[:, :, None])


In [69]:

# Inference, stats, and plotting helpers
def predict_p_fake(model: nn.Module, image: np.ndarray, transform: transforms.Compose, device: torch.device) -> float:
    pil = Image.fromarray((np.clip(image, 0.0, 1.0) * 255).astype(np.uint8))
    x = transform(pil).unsqueeze(0).to(device)
    with torch.inference_mode():
        probs = torch.softmax(model(x), dim=1)[0]
    return float(probs[1].item())


def load_samples_multi(metadata_path: Path, dataset_root: Path, split: str, class_names: list[str], per_class: int, seed: int) -> pd.DataFrame:
    meta = pd.read_csv(metadata_path)
    rows = []
    for i, cname in enumerate(class_names):
        part = meta[(meta['split'] == split) & (meta['label_name'].str.upper() == cname.upper())].copy()
        if part.empty:
            raise ValueError(f'No samples found for split={split}, class={cname}')
        part = part.sample(n=min(per_class, len(part)), random_state=seed + i).reset_index(drop=True)
        rows.append(part)
    out = pd.concat(rows, axis=0, ignore_index=True)
    out['class_name'] = out['label_name'].str.upper()
    out['abs_path'] = out['filepath'].map(lambda p: str(dataset_root / p))
    return out


def save_spectrum_figure(sample_id: str, outputs: dict[str, np.ndarray], out_dir: Path) -> None:
    fig, axes = plt.subplots(1, 4, figsize=(13, 3.5))
    axes[0].imshow(outputs['original']); axes[0].set_title('Original')
    axes[1].imshow(outputs['slight_mask']); axes[1].set_title('Modified')
    axes[2].imshow(outputs['spectrum_original'], cmap='magma'); axes[2].set_title('Original Spectrum')
    axes[3].imshow(outputs['spectrum_modified'], cmap='magma'); axes[3].set_title('Modified Spectrum')
    for ax in axes:
        ax.axis('off')
    fig.tight_layout()
    fig.savefig(out_dir / f'{sample_id}_spectrum_compare.png', dpi=180)
    plt.close(fig)


def summarize_confidence(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    g = df.groupby(group_cols, as_index=False)['p_fake'].agg(['mean', 'std', 'count']).reset_index()
    g['sem'] = g['std'] / np.sqrt(g['count'].clip(lower=1))
    return g.rename(columns={'mean': 'p_fake_mean', 'std': 'p_fake_std', 'count': 'n', 'sem': 'p_fake_sem'})


def paired_stage_tests(df: pd.DataFrame, baseline: str = 'original', by_class: bool = False) -> pd.DataFrame:
    rows = []
    class_groups = [('ALL', df)] if not by_class else list(df.groupby('class_name'))

    for class_name, part in class_groups:
        wide = part.pivot_table(index='sample_id', columns='stage', values='p_fake', aggfunc='mean')
        if baseline not in wide.columns:
            continue
        for stage in STAGES:
            if stage == baseline or stage not in wide.columns:
                continue
            paired = wide[[baseline, stage]].dropna()
            n = len(paired)
            if n < 2:
                rows.append({
                    'class_name': class_name,
                    'stage': stage,
                    'n_pairs': n,
                    'relative_drop_percent': np.nan,
                    'cohen_dz': np.nan,
                    'p_value': np.nan,
                })
                continue

            b = paired[baseline].to_numpy()
            s = paired[stage].to_numpy()
            d = s - b
            _, p_val = stats.ttest_rel(b, s)
            d_std = float(np.std(d, ddof=1))
            cohen_dz = float(np.mean(d) / d_std) if d_std > 1e-12 else np.nan
            rel_drop = float((b.mean() - s.mean()) / max(abs(b.mean()), 1e-12) * 100.0)

            rows.append({
                'class_name': class_name,
                'stage': stage,
                'n_pairs': n,
                'relative_drop_percent': rel_drop,
                'cohen_dz': cohen_dz,
                'p_value': float(p_val),
            })

    return pd.DataFrame(rows)


def save_confidence_plot(df: pd.DataFrame, out_path: Path) -> None:
    summary = summarize_confidence(df, ['stage'])
    order = [s for s in STAGES if s in summary['stage'].values]
    y = [float(summary.loc[summary['stage'] == s, 'p_fake_mean'].iloc[0]) for s in order]
    yerr = [float(summary.loc[summary['stage'] == s, 'p_fake_sem'].iloc[0]) for s in order]
    n_samples = int(df['sample_id'].nunique())

    plt.figure(figsize=(8.8, 5.2))
    plt.errorbar(order, y, yerr=yerr, marker='o', capsize=4)
    plt.ylim(0, 1)
    plt.ylabel('Mean p_fake (? SEM)')
    plt.xlabel('Pipeline stage')
    plt.title(f'Confidence Progression Across Stages (n={n_samples})')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_path, dpi=180)
    plt.close()


def save_real_vs_synthetic_plot(df: pd.DataFrame, out_path: Path) -> None:
    focus = df[df['stage'].isin(['original', 'low_pass'])].copy()
    summary = summarize_confidence(focus, ['class_name', 'stage'])
    classes = ['FAKE', 'REAL']
    stages = ['original', 'low_pass']

    x = np.arange(len(classes))
    width = 0.35
    plt.figure(figsize=(7.8, 4.8))

    for i, st in enumerate(stages):
        means = []
        sems = []
        for c in classes:
            sub = summary[(summary['class_name'] == c) & (summary['stage'] == st)]
            means.append(float(sub['p_fake_mean'].iloc[0]) if not sub.empty else np.nan)
            sems.append(float(sub['p_fake_sem'].iloc[0]) if not sub.empty else np.nan)
        plt.bar(x + (i - 0.5) * width, means, width=width, yerr=sems, capsize=4, label=st)

    plt.xticks(x, classes)
    plt.ylim(0, 1)
    plt.ylabel('Mean p_fake (? SEM)')
    plt.title('Real vs Synthetic: Original vs Low-pass Sensitivity')
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=180)
    plt.close()


def save_energy_plot(df_energy: pd.DataFrame, out_path: Path) -> None:
    summary = df_energy.groupby('stage', as_index=False)[['energy_low_ratio', 'energy_mid_ratio', 'energy_high_ratio']].mean()
    order = [s for s in STAGES if s in summary['stage'].values]
    low = [float(summary.loc[summary['stage'] == s, 'energy_low_ratio'].iloc[0]) for s in order]
    mid = [float(summary.loc[summary['stage'] == s, 'energy_mid_ratio'].iloc[0]) for s in order]
    high = [float(summary.loc[summary['stage'] == s, 'energy_high_ratio'].iloc[0]) for s in order]

    plt.figure(figsize=(8.8, 5.2))
    plt.plot(order, low, marker='o', label='Low-band ratio')
    plt.plot(order, mid, marker='o', label='Mid-band ratio')
    plt.plot(order, high, marker='o', label='High-band ratio')
    plt.ylim(0, 1)
    plt.ylabel('Mean energy ratio')
    plt.xlabel('Pipeline stage')
    plt.title('Radial Frequency Energy Ratios Across Stages')
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=180)
    plt.close()


def save_ablation_plot(df_ablation: pd.DataFrame, out_path: Path) -> None:
    order = ['remove_high', 'remove_mid', 'remove_low']
    summary = df_ablation.groupby('experiment', as_index=False)['p_fake'].agg(['mean', 'std', 'count']).reset_index()
    summary['p_fake_sem'] = summary['std'] / np.sqrt(summary['count'].clip(lower=1))

    x = [e for e in order if e in summary['experiment'].values]
    y = [float(summary.loc[summary['experiment'] == e, 'mean'].iloc[0]) for e in x]
    yerr = [float(summary.loc[summary['experiment'] == e, 'p_fake_sem'].iloc[0]) for e in x]

    plt.figure(figsize=(7.8, 4.8))
    plt.bar(x, y, yerr=yerr, capsize=4)
    plt.ylim(0, 1)
    plt.ylabel('Mean p_fake (? SEM)')
    plt.xlabel('Band removed')
    plt.title('Band-wise Ablation: Confidence Impact')
    plt.tight_layout()
    plt.savefig(out_path, dpi=180)
    plt.close()


In [70]:

# Main execution cell
set_seed(SEED)
rng = np.random.default_rng(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

model = load_model(PROJECT_ROOT, device)
tfm = model_transform(IMG_SIZE)
samples = load_samples_multi(METADATA_PATH, DATASET_ROOT, TARGET_SPLIT, TARGET_CLASSES, NUM_SAMPLES_PER_CLASS, SEED)
print(f"Selected total samples: {len(samples)}")
print(samples['class_name'].value_counts().to_dict())

rows_conf, rows_energy, rows_ablation = [], [], []

for i, row in samples.iterrows():
    img_path = Path(row['abs_path'])
    class_name = str(row['class_name'])
    sample_id = f"{class_name}_{i:03d}_{img_path.stem.replace(' ', '_')}"

    image = load_image(img_path, IMG_SIZE)
    outputs = apply_frequency_pipeline(image, CFG, rng)

    Image.fromarray((outputs['slight_mask'] * 255).astype(np.uint8)).save(MODIFIED_DIR / f'{sample_id}_modified.png')
    save_spectrum_figure(sample_id, outputs, SPECTRUM_DIR)

    for stage in STAGES:
        stage_img = outputs[stage]
        rows_conf.append({
            'sample_id': sample_id,
            'path': str(img_path),
            'class_name': class_name,
            'y_true': int(row['label']),
            'stage': stage,
            'p_fake': predict_p_fake(model, stage_img, tfm, device),
        })

        rows_energy.append({
            'sample_id': sample_id,
            'path': str(img_path),
            'class_name': class_name,
            'stage': stage,
            **radial_energy_bands(stage_img),
        })

    for exp in ABLATION_EXPERIMENTS:
        band = exp.replace('remove_', '')
        ablated = apply_band_ablation(image, band)
        rows_ablation.append({
            'sample_id': sample_id,
            'path': str(img_path),
            'class_name': class_name,
            'y_true': int(row['label']),
            'experiment': exp,
            'p_fake': predict_p_fake(model, ablated, tfm, device),
        })

# Confidence outputs
df_conf = pd.DataFrame(rows_conf)
df_conf.to_csv(OUT_DIR / 'confidence_progression.csv', index=False)

df_conf.pivot_table(index='sample_id', columns='stage', values='p_fake', aggfunc='mean').reset_index().to_csv(
    OUT_DIR / 'confidence_progression_wide.csv', index=False
)

summary_conf = summarize_confidence(df_conf, ['stage'])
summary_conf.to_csv(OUT_DIR / 'confidence_summary.csv', index=False)

summary_conf_by_class = summarize_confidence(df_conf, ['class_name', 'stage'])
summary_conf_by_class.to_csv(OUT_DIR / 'confidence_summary_by_class.csv', index=False)

save_confidence_plot(df_conf, OUT_DIR / 'confidence_progression_plot.png')

ttest_overall = paired_stage_tests(df_conf, baseline='original', by_class=False)
ttest_overall.to_csv(OUT_DIR / 'confidence_paired_ttests.csv', index=False)

# Real vs synthetic comparison (requested): original vs low-pass.
real_vs_syn = summary_conf_by_class[summary_conf_by_class['stage'].isin(['original', 'low_pass'])].copy()
real_vs_syn.to_csv(OUT_DIR / 'real_vs_synthetic_comparison.csv', index=False)
save_real_vs_synthetic_plot(df_conf, OUT_DIR / 'real_vs_synthetic_plot.png')

# Radial energy outputs
df_energy = pd.DataFrame(rows_energy)
df_energy.to_csv(OUT_DIR / 'radial_energy_progression.csv', index=False)

summary_energy = df_energy.groupby('stage', as_index=False)[['energy_low_ratio', 'energy_mid_ratio', 'energy_high_ratio']].mean()
summary_energy.to_csv(OUT_DIR / 'radial_energy_summary.csv', index=False)
save_energy_plot(df_energy, OUT_DIR / 'radial_energy_plot.png')

# Band-ablation outputs
df_ablation = pd.DataFrame(rows_ablation)
df_ablation.to_csv(OUT_DIR / 'band_ablation_confidence.csv', index=False)

summary_ablation = (
    df_ablation.groupby('experiment', as_index=False)['p_fake']
    .agg(['mean', 'std', 'count', 'min', 'max'])
    .reset_index()
    .rename(columns={'mean': 'p_fake_mean', 'std': 'p_fake_std', 'count': 'n', 'min': 'p_fake_min', 'max': 'p_fake_max'})
)
summary_ablation['p_fake_sem'] = summary_ablation['p_fake_std'] / np.sqrt(summary_ablation['n'].clip(lower=1))
summary_ablation.to_csv(OUT_DIR / 'band_ablation_summary.csv', index=False)
save_ablation_plot(df_ablation, OUT_DIR / 'band_ablation_plot.png')

print('Saved all outputs to:', OUT_DIR)


Using device: cpu
Loaded checkpoint: C:\Users\Vedha\feb hack\Feb-2026-Hackathon\Phase1\models\resnet18_subset_best.pt
Selected total samples: 48
{'FAKE': 24, 'REAL': 24}
Saved all outputs to: C:\Users\Vedha\feb hack\Feb-2026-Hackathon\Phase2\results_freq


In [71]:

# Quick summaries for report
conf_summary = pd.read_csv(OUT_DIR / 'confidence_summary.csv')
ttest_overall = pd.read_csv(OUT_DIR / 'confidence_paired_ttests.csv')
real_vs_syn = pd.read_csv(OUT_DIR / 'real_vs_synthetic_comparison.csv')
energy_summary = pd.read_csv(OUT_DIR / 'radial_energy_summary.csv')

print('Confidence summary (mean ? SEM):')
display(conf_summary)

print('Paired t-tests (overall) with effect size + relative % drop:')
display(ttest_overall)

print('Real vs Synthetic comparison (original vs low-pass):')
display(real_vs_syn)

print('Radial energy summary:')
display(energy_summary)



Confidence summary (mean ? SEM):


,index,stage,p_fake_mean,p_fake_std,n,p_fake_sem
0,0,low_pass,0.437575,0.118181,48,0.017058
1,1,original,0.497360,0.118009,48,0.017033
2,2,remove_checkerboard,0.458240,0.105411,48,0.015215
3,3,slight_mask,0.437575,0.118181,48,0.017058
4,4,suppress_high,0.496743,0.118616,48,0.017121


Paired t-tests (overall) with effect size + relative % drop:


,class_name,stage,n_pairs,relative_drop_percent,cohen_dz,p_value
0,ALL,suppress_high,48,0.124160,-0.057291,0.693221
1,ALL,remove_checkerboard,48,7.865549,-0.640874,0.000054
2,ALL,low_pass,48,12.020599,-0.549650,0.000405
3,ALL,slight_mask,48,12.020599,-0.549650,0.000405


Real vs Synthetic comparison (original vs low-pass):


,index,class_name,stage,p_fake_mean,p_fake_std,n,p_fake_sem
0,0,FAKE,low_pass,0.455081,0.097176,24,0.019836
1,1,FAKE,original,0.507112,0.100216,24,0.020457
2,5,REAL,low_pass,0.420069,0.135861,24,0.027733
3,6,REAL,original,0.487609,0.134966,24,0.027550


Radial energy summary:


,stage,energy_low_ratio,energy_mid_ratio,energy_high_ratio
0,low_pass,0.998457,0.001542,5.860355e-07
1,original,0.981166,0.017105,1.729020e-03
2,remove_checkerboard,0.991358,0.007691,9.511267e-04
3,slight_mask,0.998457,0.001542,5.860355e-07
4,suppress_high,0.982369,0.016695,9.366533e-04
